# Quantum State Simulation

Microsoft's [Quantum Katas](https://github.com/microsoft/quantumkatas)
open with a simple observation: quantum computing begins with complex
arithmetic and linear algebra.  A qubit is not a classical bit; it lives
in a two-dimensional complex vector space, and quantum gates are unitary
transformations on that space.

This notebook builds a quantum state simulator from first principles
in Gallowglass.  Because Gallowglass computes over `Nat`, we need
two new ideas before we can represent qubit amplitudes:

- **Signed fixed-point numbers** (`SFP`): a sign bit plus a scaled
  natural number, so we can represent negative and fractional values.
- **A fixed-point scale**: one constant at the top of the notebook that
  controls the decimal precision everywhere.

From those building blocks we derive:

1. `SFP` — signed fixed-point arithmetic (add, mul, negate, square)
2. `QState` — a qubit state α|0⟩ + β|1⟩ with `SFP` amplitudes
3. Measurement probabilities — |α|² and |β|²
4. `QGate` — a typeclass for quantum gates
5. Pauli-X, Pauli-Z, and Hadamard gate instances
6. Gate composition and self-inverse verification (H² ≈ I)
7. Bloch sphere coordinates — a concrete geometric picture

Assumes `05-interval-arithmetic.ipynb` — fixed-point display helpers.

## Imports

We need all of `Core.Nat`'s arithmetic: addition, multiplication, saturating subtraction, comparisons, integer division, and modulo.  `Core.Text` supplies the `Show` class and string utilities:

In [1]:
use Core.Nat unqualified { add, mul, sub, nat_lte, nat_lt, div_nat, mod_nat }
use Core.Text { Show, show, show_nat, text_concat }

use Core.Nat
use Core.Text

## Fixed-Point Scale

The irrational constant 1/√2 ≈ 0.7071… appears in every
Hadamard gate calculation.  We need at least three decimal
places of precision to track rounding errors honestly.  With
`scale = 1000` we represent 0.707 as the `Nat` value `707`
and display it as `"0.707"`.  All arithmetic stays in `Nat`;
the scale factor is the only floating-point–like thing in scope.

In [2]:
let scale : Nat = 1000

scale : Nat

In [3]:
-- floor(1000 / √2) = floor(707.106...) = 707
let inv_sqrt2_nat : Nat = 707

inv_sqrt2_nat : Nat

The same display helpers as lesson 05, now with three decimal places.  `count_digits` counts decimal digits; `frac_digits` gives the number of places implied by `scale`; `zeros` pads leading zeros; `show_fixed` assembles the decimal string:

In [4]:
let count_digits : Nat → Nat
  = λ nn →
      if nat_lt nn 10
      then 1
      else add 1 (count_digits (div_nat nn 10))

count_digits : Nat → Nat

In [5]:
let frac_digits : Nat → Nat
  = λ sc → sub (count_digits sc) 1

frac_digits : Nat → Nat

In [6]:
let zeros : Nat → Text
  = λ nn → match nn {
      | 0  → ""
      | kk → text_concat "0" (zeros kk)
    }

zeros : Nat → Text

In [7]:
let show_fixed : Nat → Nat → Text
  = λ sc nn →
      let whole = div_nat nn sc in
      let frac  = mod_nat nn sc in
      let pad   = sub (frac_digits sc) (count_digits frac) in
      text_concat (show_nat whole)
        (text_concat "."
          (text_concat (zeros pad) (show_nat frac)))

show_fixed : Nat → Nat → Text

`707` at `scale = 1000` renders as `"0.707"` — three decimal places with correct zero-padding:

In [8]:
show_fixed scale inv_sqrt2_nat

0.707

## Signed Fixed-Point Numbers (`SFP`)

A `Nat` can only represent zero or positive values.  Quantum
amplitudes can be negative (e.g. the |−⟩ state has β = −1/√2).
We introduce a two-constructor `Sign` type to carry the sign
bit, then pack it with a `Nat` magnitude into `SFP`.

`MkSFP Pos 707` represents +0.707.  `MkSFP Neg 707` represents
−0.707.  The `Nat` field is always the *magnitude* — never
a two's-complement encoding.

In [9]:
type Sign = | Pos | Neg

type Sign = Pos | Neg

In [10]:
type SFP = | MkSFP Sign Nat

type SFP = MkSFP

Three fundamental constants.  `sfp_zero` is 0.000; `sfp_one`
is 1.000 (i.e. one `scale` unit); `inv_sqrt2` is +0.707:

In [11]:
let sfp_zero : SFP = MkSFP Pos 0

sfp_zero : Notebook.SFP

In [12]:
let sfp_one  : SFP = MkSFP Pos scale

sfp_one : Notebook.SFP

In [13]:
let inv_sqrt2 : SFP = MkSFP Pos inv_sqrt2_nat

inv_sqrt2 : Notebook.SFP

## SFP Arithmetic

`sfp_neg` flips the sign bit.  Negating `Pos 0` yields
`Neg 0` — a signed zero — which compares as zero in all
subsequent arithmetic:

In [14]:
let sfp_neg : SFP → SFP
  = λ ss →
      match ss {
        | MkSFP sg mm →
            match sg {
              | Pos → MkSFP Neg mm
              | Neg → MkSFP Pos mm
            }
      }

sfp_neg : Notebook.SFP → Notebook.SFP

`sfp_mul_sign` encodes the sign-of-product rule:
(+)(+) = (+); (+)(−) = (−); (−)(−) = (+):

In [15]:
let sfp_mul_sign : Sign → Sign → Sign
  = λ sa sb →
      match sa {
        | Pos → match sb { | Pos → Pos | Neg → Neg }
        | Neg → match sb { | Pos → Neg | Neg → Pos }
      }

sfp_mul_sign : Notebook.Sign → Notebook.Sign → Notebook.Sign

`sfp_add` implements sign-magnitude addition.  When the signs
agree, add the magnitudes and keep the sign.  When they differ,
subtract the smaller from the larger and take the sign of the
larger:

```
(+a) + (+b) = +(a + b)
(+a) + (−b) = +(a − b) if b ≤ a, else −(b − a)
(−a) + (+b) = +(b − a) if a ≤ b, else −(a − b)
(−a) + (−b) = −(a + b)
```

In [16]:
let sfp_add : SFP → SFP → SFP
  = λ aa bb →
      match aa {
        | MkSFP sa ma →
            match bb {
              | MkSFP sb mb →
                  match sa {
                    | Pos →
                        match sb {
                          | Pos → MkSFP Pos (add ma mb)
                          | Neg →
                              if nat_lte mb ma
                              then MkSFP Pos (sub ma mb)
                              else MkSFP Neg (sub mb ma)
                        }
                    | Neg →
                        match sb {
                          | Pos →
                              if nat_lte ma mb
                              then MkSFP Pos (sub mb ma)
                              else MkSFP Neg (sub ma mb)
                          | Neg → MkSFP Neg (add ma mb)
                        }
                  }
            }
      }

sfp_add : Notebook.SFP → Notebook.SFP → Notebook.SFP

`sfp_mul` multiplies two fixed-point values.  Multiplying two
scale-1000 numbers gives a scale-1000000 product; dividing by
`scale` brings it back to scale-1000:

```
(a/1000) × (b/1000) = (a×b)/1000000 = (a×b÷1000)/1000
```

In [17]:
let sfp_mul : SFP → SFP → SFP
  = λ aa bb →
      match aa {
        | MkSFP sa ma →
            match bb {
              | MkSFP sb mb →
                  MkSFP (sfp_mul_sign sa sb)
                        (div_nat (mul ma mb) scale)
            }
      }

sfp_mul : Notebook.SFP → Notebook.SFP → Notebook.SFP

`sfp_sq` squares an `SFP`.  The result is always positive
(sign² = +), so we can skip the sign dispatch:

In [18]:
let sfp_sq : SFP → SFP
  = λ ss →
      match ss {
        | MkSFP sg mm → MkSFP Pos (div_nat (mul mm mm) scale)
      }

sfp_sq : Notebook.SFP → Notebook.SFP

## Displaying `SFP` Values

`show_sfp` renders the sign as `"+"` or `"-"` and appends
the fixed-point decimal.  Positive values carry an explicit
`"+"` so every amplitude in a state vector has a visible sign:

In [19]:
let show_sfp : SFP → Text
  = λ ss →
      match ss {
        | MkSFP sg mm →
            let sign_str =
              match sg { | Pos → "+" | Neg → "-" } in
            text_concat sign_str (show_fixed scale mm)
      }

show_sfp : Notebook.SFP → Text

A quick sanity check — zero, one, and the two signs of 1/√2:

In [20]:
show_sfp sfp_zero

+0.000

In [21]:
show_sfp sfp_one

+1.000

In [22]:
show_sfp inv_sqrt2

+0.707

In [23]:
show_sfp (sfp_neg inv_sqrt2)

-0.707

## Qubit State — α|0⟩ + β|1⟩

A pure qubit state is a superposition of the two computational
basis states |0⟩ and |1⟩ with complex amplitudes α and β
satisfying |α|² + |β|² = 1.

We restrict to *real* amplitudes in this tutorial — complex
amplitudes require two `SFP` components per amplitude and are
the natural extension once you want the Y gate or arbitrary
phase gates.  Real amplitudes are enough for X, Z, and H.

`MkQState alpha beta` holds the amplitude pair:

In [24]:
type QState = | MkQState SFP SFP

type QState = MkQState

The four named states.  `ket_zero` and `ket_one` are the
computational basis; `ket_plus` and `ket_minus` are the
diagonal basis (eigenstates of the Hadamard gate):

In [25]:
let ket_zero  : QState = MkQState sfp_one sfp_zero

ket_zero : Notebook.QState

In [26]:
let ket_one   : QState = MkQState sfp_zero sfp_one

ket_one : Notebook.QState

In [27]:
let ket_plus  : QState = MkQState inv_sqrt2 inv_sqrt2

ket_plus : Notebook.QState

In [28]:
let ket_minus : QState = MkQState inv_sqrt2 (sfp_neg inv_sqrt2)

ket_minus : Notebook.QState

`show_qs` renders a state as `±A.AAA|0⟩ + ±B.BBB|1⟩`:

In [29]:
let show_qs : QState → Text
  = λ qs →
      match qs {
        | MkQState alpha beta →
            text_concat (show_sfp alpha)
              (text_concat "|0⟩ + "
                (text_concat (show_sfp beta) "|1⟩"))
      }

show_qs : Notebook.QState → Text

In [30]:
show_qs ket_zero

+1.000|0⟩ + +0.000|1⟩

In [31]:
show_qs ket_one

+0.000|0⟩ + +1.000|1⟩

In [32]:
show_qs ket_plus

+0.707|0⟩ + +0.707|1⟩

In [33]:
show_qs ket_minus

+0.707|0⟩ + -0.707|1⟩

## Measurement Probabilities

Measuring a qubit in state α|0⟩ + β|1⟩ yields |0⟩ with
probability |α|² and |1⟩ with probability |β|².  In our
scale-1000 representation, `prob_zero` and `prob_one` return
values in [0, 1000] where 1000 means certainty (100.0 %)
and 499 means ≈ 49.9 %.

`sfp_prob` extracts the magnitude and squares it (sign is
irrelevant — probability is always non-negative):

In [34]:
let sfp_prob : SFP → Nat
  = λ ss →
      match ss {
        | MkSFP sg mm → div_nat (mul mm mm) scale
      }

sfp_prob : Notebook.SFP → Nat

In [35]:
let prob_zero : QState → Nat
  = λ qs → match qs { | MkQState alpha beta → sfp_prob alpha }

prob_zero : Notebook.QState → Nat

In [36]:
let prob_one : QState → Nat
  = λ qs → match qs { | MkQState alpha beta → sfp_prob beta }

prob_one : Notebook.QState → Nat

`ket_zero` gives 100.0 % probability of measuring |0⟩.
`ket_plus` gives ≈ 49.9 % — the fixed-point error from
707² ÷ 1000 = 499.849 → 499 is 0.1 %:

In [37]:
prob_zero ket_zero

1000

In [38]:
prob_zero ket_plus

499

The Born rule requires |α|² + |β|² = 1.  At scale 1000 the
two probabilities sum to 998 rather than 1000 — the 0.2 %
shortfall from rounding 1/√2 to three decimal places.  This
is the honest price of fixed-point simulation:

In [39]:
add (prob_zero ket_plus) (prob_one ket_plus)

998

## The `QGate` Typeclass

A quantum gate is any type `g` that can transform a `QState`.
`QGate` captures exactly that contract:

```
class QGate g {
  apply_gate : g → QState → QState
}
```

Three instances follow for the Pauli-X, Pauli-Z, and Hadamard
gates — the simplest complete set from the Quantum Katas
introductory module.

In [40]:
class QGate g {
  apply_gate : g → QState → QState
}

## Pauli-X Gate (Bit Flip)

X|0⟩ = |1⟩, X|1⟩ = |0⟩.  In the amplitude representation,
X simply swaps α and β.  The matrix is:

```
X = | 0  1 |
    | 1  0 |
```

In [41]:
type GateX = | GateX

type GateX = GateX

In [42]:
instance QGate GateX {
  apply_gate = λ gg qs →
      match qs {
        | MkQState alpha beta → MkQState beta alpha
      }
}

## Pauli-Z Gate (Phase Flip)

Z|0⟩ = |0⟩, Z|1⟩ = −|1⟩.  Z leaves the |0⟩ amplitude
unchanged and negates the |1⟩ amplitude:

```
Z = | 1   0 |
    | 0  −1 |
```

Z turns |+⟩ into |−⟩ and vice versa — it is the bit-flip
gate in the *diagonal* (Hadamard) basis.

In [43]:
type GateZ = | GateZ

type GateZ = GateZ

In [44]:
instance QGate GateZ {
  apply_gate = λ gg qs →
      match qs {
        | MkQState alpha beta → MkQState alpha (sfp_neg beta)
      }
}

## Hadamard Gate (Superposition)

The Hadamard gate creates — or collapses — superposition.
H|0⟩ = |+⟩, H|1⟩ = |−⟩, H|+⟩ ≈ |0⟩, H|−⟩ ≈ |1⟩.

```
H = (1/√2) | 1   1 |
            | 1  −1 |
```

Applied to α|0⟩ + β|1⟩:

```
α′ = (α + β) / √2
β′ = (α − β) / √2
```

In `SFP`: compute `sum = α + β` and `diff = α − β`, then
multiply each by `inv_sqrt2` (= 0.707).  Fixed-point
multiplication loses ≈ 0.1 % per application — two
applications accumulate to H² ≈ I with 0.1 % residual:

In [45]:
type GateH = | GateH

type GateH = GateH

In [46]:
instance QGate GateH {
  apply_gate = λ gg qs →
      match qs {
        | MkQState alpha beta →
            let sum  = sfp_add alpha beta in
            let diff = sfp_add alpha (sfp_neg beta) in
            MkQState (sfp_mul sum  inv_sqrt2)
                     (sfp_mul diff inv_sqrt2)
      }
}

The constrained wrapper routes through the typeclass dictionary.
The `∀ g. QGate g =>` quantifier is what triggers dictionary
insertion at compile time:

In [47]:
let run_gate : ∀ g. QGate g => g → QState → QState
  = λ gg qs → apply_gate gg qs

run_gate : ∀ g. g → Notebook.QState → Notebook.QState

## Gate Demonstrations

`X|0⟩` flips the bit — we get |1⟩:

In [48]:
show_qs (run_gate GateX ket_zero)

+0.000|0⟩ + +1.000|1⟩

`H|0⟩` creates equal superposition — we get |+⟩:

In [49]:
show_qs (run_gate GateH ket_zero)

+0.707|0⟩ + +0.707|1⟩

`H|1⟩` creates |−⟩ (equal superposition, negative β):

In [50]:
show_qs (run_gate GateH ket_one)

+0.707|0⟩ + -0.707|1⟩

`Z|+⟩` flips the phase — we get |−⟩.  This is the Pauli-X
gate in the Hadamard basis: just as X flips |0⟩ ↔ |1⟩, Z
flips |+⟩ ↔ |−⟩:

In [51]:
show_qs (run_gate GateZ ket_plus)

+0.707|0⟩ + -0.707|1⟩

## Self-Inverse Verification

All three gates are their own inverses: G² = I.

X² is exact — swapping twice is a no-op:

In [52]:
show_qs (run_gate GateX (run_gate GateX ket_plus))

+0.707|0⟩ + +0.707|1⟩

Z² is exact — negating twice is a no-op:

In [53]:
show_qs (run_gate GateZ (run_gate GateZ ket_plus))

+0.707|0⟩ + +0.707|1⟩

H² accumulates fixed-point rounding: `707 × 707 ÷ 1000 = 499`,
and `(707 + 707) × 707 ÷ 1000 = 999`.  The α amplitude of
H(H|0⟩) is `+0.999`, not `+1.000`.  The 0.1 % error is the
honest cost of approximating 1/√2 as 0.707:

In [54]:
show_qs (run_gate GateH (run_gate GateH ket_zero))

+0.999|0⟩ + +0.000|1⟩

## Bloch Sphere Coordinates

Every pure qubit state corresponds to a point on the unit
sphere (the Bloch sphere).  For real amplitudes α, β:

```
x = 2αβ          (longitude)
z = α² − β²      (latitude: +1 = north pole = |0⟩,
                            −1 = south pole = |1⟩)
```

|0⟩ sits at the north pole (x=0, z=+1); |1⟩ at the south
pole (x=0, z=−1); |+⟩ and |−⟩ on the equator at ±x.

`sfp_two` is the fixed-point representation of 2.000:

In [55]:
let sfp_two : SFP = MkSFP Pos (mul 2 scale)

sfp_two : Notebook.SFP

In [56]:
let bloch_z : QState → SFP
  = λ qs →
      match qs {
        | MkQState alpha beta →
            sfp_add (sfp_sq alpha) (sfp_neg (sfp_sq beta))
      }

bloch_z : Notebook.QState → Notebook.SFP

In [57]:
let bloch_x : QState → SFP
  = λ qs →
      match qs {
        | MkQState alpha beta →
            sfp_mul sfp_two (sfp_mul alpha beta)
      }

bloch_x : Notebook.QState → Notebook.SFP

The four basis states at their expected Bloch sphere positions:

In [58]:
text_concat "ket_zero  z=" (text_concat (show_sfp (bloch_z ket_zero))
  (text_concat "  x=" (show_sfp (bloch_x ket_zero))))

ket_zero  z=+1.000  x=+0.000

In [59]:
text_concat "ket_one   z=" (text_concat (show_sfp (bloch_z ket_one))
  (text_concat "  x=" (show_sfp (bloch_x ket_one))))

ket_one   z=-1.000  x=+0.000

In [60]:
text_concat "ket_plus  z=" (text_concat (show_sfp (bloch_z ket_plus))
  (text_concat "  x=" (show_sfp (bloch_x ket_plus))))

ket_plus  z=+0.000  x=+0.998

In [61]:
text_concat "ket_minus z=" (text_concat (show_sfp (bloch_z ket_minus))
  (text_concat "  x=" (show_sfp (bloch_x ket_minus))))

ket_minus z=+0.000  x=-0.998

`ket_plus` has x ≈ +0.998 (should be +1.000): the 0.2 %
shortfall comes from 707² ÷ 1000 = 499 rather than 500.
Bloch coordinates are quadratic in the amplitudes, so the
fixed-point error is doubled vs. the amplitude error.

Applying a gate rotates the point on the sphere.  X is a π
rotation about the X-axis (north ↔ south); Z is a π rotation
about the Z-axis (|+⟩ ↔ |−⟩); H is a π/2 rotation that
maps |0⟩ → |+⟩:

In [62]:
text_concat "H|0⟩  z=" (text_concat (show_sfp (bloch_z (run_gate GateH ket_zero)))
  (text_concat "  x=" (show_sfp (bloch_x (run_gate GateH ket_zero)))))

H|0⟩  z=+0.000  x=+0.000

In [63]:
text_concat "X|0⟩  z=" (text_concat (show_sfp (bloch_z (run_gate GateX ket_zero)))
  (text_concat "  x=" (show_sfp (bloch_x (run_gate GateX ket_zero)))))

X|0⟩  z=-1.000  x=+0.000

## Dominant Measurement Outcome

Given a state, which basis state is more likely to be measured?
`more_zero` returns `True` when P(|0⟩) > P(|1⟩).
For |+⟩, the probabilities are equal (499 = 499) so
`more_zero` returns `False` — not *strictly* more likely:

In [64]:
let more_zero : QState → Bool
  = λ qs →
      match qs {
        | MkQState alpha beta →
            nat_lt (sfp_prob beta) (sfp_prob alpha)
      }

more_zero : Notebook.QState → Bool

In [65]:
more_zero ket_zero

True

In [66]:
more_zero ket_one

False

In [67]:
more_zero ket_plus

False

After an H gate the computational basis state |0⟩ becomes
an equal superposition — so even |0⟩, which starts with
100 % probability of measuring |0⟩, loses that certainty
the moment it enters a Hadamard:

In [68]:
more_zero (run_gate GateH ket_zero)

False

## Summary

We built a single-qubit quantum state simulator from scratch in
Gallowglass using only `Nat` arithmetic.

| Component | What it is |
|-----------|------------|
| `SFP` | Signed fixed-point number: sign bit + scaled `Nat` |
| `QState` | Qubit state: two `SFP` amplitudes α, β |
| `QGate` | Typeclass: any type that can transform a `QState` |
| `GateX` | Pauli-X (bit flip): α ↔ β |
| `GateZ` | Pauli-Z (phase flip): β ↦ −β |
| `GateH` | Hadamard: α′ = (α+β)/√2, β′ = (α−β)/√2 |

The fixed-point rounding errors (0.1 % per H application) are not
a flaw to be hidden — they are the honest signature of finite precision,
and understanding them is a prerequisite for reasoning about numerical
quantum simulation.  A scale of 10 000 would cut the error tenfold;
exact rational arithmetic would eliminate it entirely.

Natural extensions:
- **Complex amplitudes**: add a second `SFP` field per amplitude
  (`MkComplex SFP SFP`) to support the Y gate and arbitrary phase rotations.
- **Two-qubit states**: tensor-product representation as four `SFP`
  amplitudes for |00⟩, |01⟩, |10⟩, |11⟩; CNOT becomes a 4×4 gate.
- **Grover's algorithm**: an oracle gate instance plus the diffusion
  operator, both as `QGate` instances on a two-qubit `QState`.